In [ ]:
import pandas as pd
import pyarrow.parquet as pq
from neo4j import GraphDatabase
import time
import os
from dotenv import load_dotenv

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
load_dotenv() 

URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USER")
PASSWORD = os.getenv("NEO4J_PASSWORD")
PARQUET_FILE = 'part-00000-f078acc1-ab56-40a6-a6e1-99d780645c57-c000.snappy.parquet'

print("🔍 Iniciando test de diagnóstico...")

try:
    print("1️⃣ Abriendo archivo Parquet...")
    parquet_file = pq.ParquetFile(PARQUET_FILE)
    print("✅ Archivo abierto. Leyendo metadatos...")
    print(f"📊 Total de Row Groups en el archivo: {parquet_file.num_row_groups}")

    print("2️⃣ Intentando extraer el primer lote (1.000 filas)...")
    # Pedimos solo 1000 para forzar a pyarrow a trabajar poco
    batches = parquet_file.iter_batches(batch_size=1000)
    primer_lote = next(batches)
    print("✅ PyArrow logró extraer el lote. ¡El archivo NO es el problema!")

    print("3️⃣ Limpiando datos...")
    df = primer_lote.to_pandas()
    # Hacemos una limpieza mínima solo para que no falle la query
    df['orig_bytes'] = pd.to_numeric(df['orig_bytes'], errors='coerce').fillna(0)
    df['resp_bytes'] = pd.to_numeric(df['resp_bytes'], errors='coerce').fillna(0)
    df['orig_pkts'] = pd.to_numeric(df['orig_pkts'], errors='coerce').fillna(0)
    df['resp_pkts'] = pd.to_numeric(df['resp_pkts'], errors='coerce').fillna(0)
    df = df.fillna('N/A')
    batch_data = df.to_dict('records')

    print("4️⃣ Enviando datos a Neo4j...")
    
    query = """
    UNWIND $batch AS row
    MERGE (src:IP {address: row.src_ip_zeek})
    MERGE (dst:IP {address: row.dest_ip_zeek})
    MERGE (src)-[c:CONNECTED_TO {uid: row.uid}]->(dst)
    """
    
    with GraphDatabase.driver(URI, auth=(USER, PASSWORD)) as driver:
        with driver.session() as session:
            session.run(query, batch=batch_data)
            
    print("✅ ¡Neo4j ha tragado los datos! El problema era otro.")

except Exception as e:
    print(f"❌ El script ha petado y el error es: {e}")